In [1]:
import pandas as pd
import numpy as np

#### 2.1
Calculate EW Covariance, lambda=0.97 using test2.csv and testout_2.1.csv

In [20]:
# read input data
data = pd.read_csv('../testfiles/data/test2.csv')

In [3]:
lambda_ = 0.97   # how fast cov decays
alpha = 1 - lambda_

In [19]:
ew_cov = data.ewm(alpha=alpha, adjust=True).cov()

# most recent cov
ew_cov_unbiased = ew_cov.loc[data.index[-1]]  # sample cov
ew_cov_unbiased

,x1,x2,x3,x4,x5
x1,0.873091,0.123503,0.184313,0.086361,0.057030
x2,0.123503,1.110707,0.025394,0.118442,-0.441995
x3,0.184313,0.025394,0.757684,0.137621,0.071023
x4,0.086361,0.118442,0.137621,0.892950,0.116039
x5,0.057030,-0.441995,0.071023,0.116039,1.170727


In [17]:
# correction factor: 1 - sum(weights^2), weights[i] = alpha * lambda^i
def correct_factor(data):
    n = len(data)
    w = np.array([alpha * lambda_**i for i in range(n)])
    w /= w.sum()
    correction_factor = 1 - np.sum(w**2)
    return correction_factor

In [18]:
correction_factor = correct_factor(data)

ew_cov_biased = ew_cov_unbiased * correction_factor   # pop cov
ew_cov_biased

,x1,x2,x3,x4,x5
x1,0.848630,0.120043,0.179149,0.083942,0.055432
x2,0.120043,1.079589,0.024682,0.115124,-0.429611
x3,0.179149,0.024682,0.736457,0.133765,0.069033
x4,0.083942,0.115124,0.133765,0.867932,0.112788
x5,0.055432,-0.429611,0.069033,0.112788,1.137927


#### 2.2
Calculate EW Correlation, lambd=0.94 using test2.csv and testout_2.2.csv

In [5]:
lambda_ = 0.94
alpha = 1 - lambda_

In [6]:
ew_corr = data.ewm(alpha=alpha).corr()

# most recent cov
ew_corr_last = ew_corr.loc[data.index[-1]]
ew_corr_last

,x1,x2,x3,x4,x5
x1,1.000000,0.084787,0.190100,0.129045,0.070706
x2,0.084787,1.000000,-0.077432,0.202228,-0.442546
x3,0.190100,-0.077432,1.000000,0.198267,0.102093
x4,0.129045,0.202228,0.198267,1.000000,0.119541
x5,0.070706,-0.442546,0.102093,0.119541,1.000000


#### 2.3
Calculate Covariance with EW Variance (lambda=0.97) and EW Correlation (lambda=0.94) using test2.csv and testout_2.3.csv

In [7]:
lambda_ = 0.97
alpha = 1 - lambda_

In [27]:
# cov: Σ = D * R * D, where D = diag(EW std), R = EW corr
ew_var = data.ewm(alpha=alpha).var()
var_last = ew_var.loc[data.index[-1]]
var_biased = var_last * correction_factor

ew_sd = np.sqrt(var_biased)

D = np.diag(ew_sd)
D

array([[0.92121134, 0.        , 0.        , 0.        , 0.        ],
       [0.        , 1.03903277, 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.85817049, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.93162893, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 1.06673668]])

In [28]:
cov = D @ ew_corr_last @ D
cov

,0,1,2,3,4
0,0.848630,0.081156,0.150285,0.110750,0.069482
1,0.081156,1.079589,-0.069043,0.195755,-0.490507
2,0.150285,-0.069043,0.736457,0.158514,0.093460
3,0.110750,0.195755,0.158514,0.867932,0.118800
4,0.069482,-0.490507,0.093460,0.118800,1.137927
